In [1]:
import pandas as pd

# csv als DataFrame einlesen
df = pd.read_csv("project/survey_results_public.csv", index_col="ResponseId")

#Spalten mit SO entfernen
df = df.drop(columns=df.columns[df.columns.str.startswith('SO')])

#EdLevel splitten um nur EdLevel anzuzeigen
df["EdLevel"] = df["EdLevel"].str.split("(").str[0].str.strip()

#RemoteWork auf numerische Werte mappen
remote_map = {
    "Remote": 0,
    "In-person": 1,
    "Hybrid (some remote, leans heavy to in-person)": 0.75,
    "Hybrid (some in-person, leans heavy to flexibility)": 0.25,
    "Your choice (very flexible, you can come in when you want or just as needed)": 0.5
}
df.insert(
    df.columns.get_loc("RemoteWork") + 1,
    "RemoteCategoryNum",
    df["RemoteWork"].map(remote_map)
)

#Age zu numerischen Werten mappen, immer Mittelwert der ranges
age_map = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29,
    "35-44 years old": 39,
    "45-54 years old": 49,
    "55-64 years old": 59,
    "65 years or older": 70
}

df.insert(
    df.columns.get_loc("Age") + 1,
    "AgeNum",
    df['Age'].map(age_map)
)

#Age zu numerischen Werten mappen, immer oberer Range
age_map2 = {
    "Under 18 years old": 18,
    "18-24 years old": 24,
    "25-34 years old": 34,
    "35-44 years old": 44,
    "45-54 years old": 54,
    "55-64 years old": 64,
    "65 years or older": 100
}

df.insert(
    df.columns.get_loc("Age") + 1,
    "MaxAge",
    df['Age'].map(age_map2)
)


#Über 70 jährige entfernen
df = df[df['AgeNum'] <= 65]


df.to_csv("survey_results_shortened.csv", index=False)


# 🧹 Datenbereinigung & Vereinheitlichung – To-Do Liste

Diese To-Do-Liste beschreibt alle notwendigen Schritte, um die Survey-CSV-Datei zu bereinigen, zu vereinheitlichen und für spätere Analysen oder Visualisierungen nutzbar zu machen.

---

## 1. Fehlende Werte standardisieren

### Kategoriale Spalten
- `NaN` → **"Keine Angabe"**
- Datentyp `object` beibehalten

### Numerische Spalten
- `NaN` **nicht ersetzen**
- Datentyp `int`/`float` belassen
  → wichtig für statistische Auswertungen (Durchschnitt, Median, Histogramme)

In [2]:
import unicodedata
import re

In [3]:
# Alle Spalten mit numerischen Werten in einen DataFrame packen
numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Alle Spalten mit nicht numerischen Werten in einen DataFrame packen
category_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Anzahl numerischer Spalten:", len(numeric_columns)) #Anzahl: 42
print("Anzahl kategorischer Spalten:", len(category_columns)) #Anzahl: 106

print("\nBeispiele numerischer Spalten:", numeric_columns[:10])
print("\nBeispiele kategorischer Spalten:", category_columns[:10])

Anzahl numerischer Spalten: 43
Anzahl kategorischer Spalten: 106

Beispiele numerischer Spalten: ['MaxAge', 'AgeNum', 'WorkExp', 'YearsCode', 'RemoteCategoryNum', 'TechEndorse_1', 'TechEndorse_2', 'TechEndorse_3', 'TechEndorse_4', 'TechEndorse_5']

Beispiele kategorischer Spalten: ['MainBranch', 'Age', 'EdLevel', 'Employment', 'EmploymentAddl', 'LearnCodeChoose', 'LearnCode', 'LearnCodeAI', 'AILearnHow', 'DevType']


## 2. Kategorische Text-Spalten bereinigen

### Ziel
Eine einheitliche und konsistente Darstellung, um spätere Gruppierungen und Analysen zu erleichtern.

### Maßnahmen
- Whitespace entfernen (Trimmen)
- Einheitliche Groß-/Kleinschreibung (z. B. `title()` oder `lower()`)
- Zusammenführen identischer kategorischer Werte mit unterschiedlicher Schreibweise
  *Beispiel: „Self taught“ und „self-taught“*
- Optional: Seltene Kategorien in **"Other"** gruppieren

### Beispiele betroffener Spalten
- `MainBranch`
- `EdLevel`
- `Employment`
- `Country`
- `OrgSize`
- `Industry`
- `AISelect`
- `AIPrimaryUse`

In [4]:
# Normalisierungs-/Vereinheitlichungsfunktion für Strings zur Anwendung in nicht numerischen Spalten
def clean_text(s):
    if pd.isna(s):
        return s
    s = str(s).strip() # Leerzeichen am Anfang und Ende weg
    s = unicodedata.normalize("NFC", s) # Unicode normalisieren
    s = s.replace("–", "-").replace("—", "-")
    s = s.replace("’", "'") # Apostrophen vereinheitlichen
    s = re.sub(r"\s+", " ", s)
    return s

In [5]:
# Alles in der Spalte zu lowercase ändern um besser damit zu arbeiten
def to_lowercase(s):
    if pd.isna(s):
        return s
    return s.lower()

In [6]:
# Anwendung der Normalisierungs-/Vereinheitlichungsfunktion auf alle nicht numerischen Spalten
for col in category_columns:
    df[col] = df[col].apply(clean_text)

In [7]:
# Currency außen vor wegen den Währungscodes wie USD oder EUR
exclude_columns = ["Country", "Currency"]

for col in category_columns:
    if col not in exclude_columns:
        df[col] = df[col].apply(to_lowercase)

## 3. Mehrfachauswahl-Spalten vereinheitlichen (`;`-getrennte Werte)

### Typische Probleme
- Uneinheitliche Formatierungen
- Semikolon-separierte Werte
- NaN-Werte
- Inkonsistente Reihenfolgen

### Maßnahmen
- Aufsplitten in Listen
  `"Python; JavaScript"` → `["Python", "JavaScript"]`
- Werte trimmen
- Duplikate in Listen entfernen
- Optionale alphabetische Sortierung der Werte
- NaN → **leere Liste** oder **"Keine Angabe"**

### Beispiele
- `DevType`
- `LanguageHaveWorkedWith`
- `LanguageWantToWorkWith`
- `DatabaseHaveWorkedWith`
- `ToolsTechHaveWorkedWith`
- `PlatformHaveWorkedWith`

In [8]:
# Ausgabe aller Spalten, die Semikolon-getrennt sind
[col for col in df.columns if df[col].astype(str).str.contains(";").any()]

['EmploymentAddl',
 'LearnCode',
 'AILearnHow',
 'TechEndorse_13_TEXT',
 'TechOppose_15_TEXT',
 'JobSatPoints_15_TEXT',
 'LanguageHaveWorkedWith',
 'LanguageWantToWorkWith',
 'LanguageAdmired',
 'LanguagesHaveEntry',
 'LanguagesWantEntry',
 'DatabaseHaveWorkedWith',
 'DatabaseWantToWorkWith',
 'DatabaseAdmired',
 'DatabaseHaveEntry',
 'DatabaseWantEntry',
 'PlatformHaveWorkedWith',
 'PlatformWantToWorkWith',
 'PlatformAdmired',
 'PlatformWantEntry',
 'WebframeHaveWorkedWith',
 'WebframeWantToWorkWith',
 'WebframeAdmired',
 'WebframeHaveEntry',
 'WebframeWantEntry',
 'DevEnvsHaveWorkedWith',
 'DevEnvsWantToWorkWith',
 'DevEnvsAdmired',
 'DevEnvHaveEntry',
 'DevEnvWantEntry',
 'OpSysPersonal use',
 'OpSysProfessional use',
 'OfficeStackAsyncHaveWorkedWith',
 'OfficeStackAsyncWantToWorkWith',
 'OfficeStackAsyncAdmired',
 'OfficeStackHaveEntry',
 'CommPlatformHaveWorkedWith',
 'CommPlatformWantToWorkWith',
 'CommPlatformAdmired',
 'CommPlatformHaveEntr',
 'CommPlatformWantEntr',
 'AIModels

In [9]:
# Alle Semikolon-getrennten Spalten in einer Liste
multi_select_cols = [
 'EmploymentAddl','LearnCode','AILearnHow','TechEndorse_13_TEXT','TechOppose_15_TEXT',
 'JobSatPoints_15_TEXT','LanguageHaveWorkedWith','LanguageWantToWorkWith','LanguageAdmired',
 'LanguagesHaveEntry','LanguagesWantEntry','DatabaseHaveWorkedWith','DatabaseWantToWorkWith',
 'DatabaseAdmired','DatabaseHaveEntry','DatabaseWantEntry','PlatformHaveWorkedWith',
 'PlatformWantToWorkWith','PlatformAdmired','PlatformWantEntry','WebframeHaveWorkedWith',
 'WebframeWantToWorkWith','WebframeAdmired','WebframeHaveEntry','WebframeWantEntry',
 'DevEnvsHaveWorkedWith','DevEnvsWantToWorkWith','DevEnvsAdmired','DevEnvHaveEntry',
 'DevEnvWantEntry','OpSysPersonal use','OpSysProfessional use',
 'OfficeStackAsyncHaveWorkedWith','OfficeStackAsyncWantToWorkWith','OfficeStackAsyncAdmired',
 'OfficeStackHaveEntry','CommPlatformHaveWorkedWith','CommPlatformWantToWorkWith',
 'CommPlatformAdmired','CommPlatformHaveEntr','CommPlatformWantEntr',
 'AIModelsHaveWorkedWith','AIModelsWantToWorkWith','AIModelsAdmired',
 'AIToolCurrently partially AI',"AIToolDon't plan to use AI for this task",
 'AIToolPlan to partially use AI','AIToolPlan to mostly use AI','AIToolCurrently mostly AI',
 'AIFrustration','AIExplain','AIAgent_Uses','AgentUsesGeneral',
 'AIAgentImpactSomewhat agree','AIAgentImpactNeutral','AIAgentImpactSomewhat disagree',
 'AIAgentImpactStrongly agree','AIAgentImpactStrongly disagree',
 'AIAgentChallengesNeutral','AIAgentChallengesSomewhat disagree',
 'AIAgentChallengesStrongly agree','AIAgentChallengesSomewhat agree',
 'AIAgentChallengesStrongly disagree','AIAgentKnowledge','AIAgentKnowWrite',
 'AIAgentOrchestration','AIAgentOrchWrite','AIAgentObserveSecure','AIAgentObsWrite',
 'AIAgentExternal','AIAgentExtWrite','AIHuman','AIOpen'
]

In [10]:
def clean_multi_select(value):
    if pd.isna(value):
        return []

    splitted = str(value).split(";")

    cleaned = []
    for p in splitted:
        clean_text(p)
        if p:
            cleaned.append(p)

    cleaned = list(set(cleaned))

    cleaned.sort()

    return cleaned

In [11]:
for col in multi_select_cols:
    df[col] = df[col].apply(clean_multi_select)

In [12]:
df['YearsCode']

ResponseId
1        14.0
2        10.0
3        12.0
4         5.0
5        22.0
         ... 
49119    13.0
49120    15.0
49121     NaN
49122    14.0
49123    15.0
Name: YearsCode, Length: 47803, dtype: float64

In [13]:
df.to_csv("survey_results_shortened.csv", index=False)


## Filter die Zeilen raus, die mehr Berufs-Erfahrung haben als Lebensalter - 16 Jahre

In [14]:
# Keep only rows where WorkExp is NOT greater than (MaxAge - 16)
filtered_df = df[~(df['WorkExp'] > (df['MaxAge'] - 16))]

# Show the number of rows before and after filtering
print(f"Original number of rows: {len(df)}")
print(f"Number of rows after filtering: {len(filtered_df)}")
print(f"Removed {len(df) - len(filtered_df)} rows ({(1 - len(filtered_df)/len(df))*100:.1f}%)")

# Display the first few rows of the filtered dataframe
filtered_df.head()

Original number of rows: 47803
Number of rows after filtering: 47553
Removed 250 rows (0.5%)


,MainBranch,Age,MaxAge,AgeNum,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
ResponseId,,,,,,,,,,,,,,,,,,,,,
1,i am a developer by profession,25-34 years old,34.0,29.0,master's degree,employed,"[caring for dependents (children, elderly, etc.)]",8.0,"yes, i am not new to coding but am learning ne...",[online courses or certification (includes all...,...,[vertex ai],[],[],[],[chatgpt],[],[when i don't trust ai's answers],"[troubleshooting, profiling, debugging]",61256.0,10.0
2,i am a developer by profession,25-34 years old,34.0,29.0,associate degree,employed,[],2.0,"yes, i am not new to coding but am learning ne...","[books / physical media, online courses or cer...",...,[],[],[],[],[],[],"[when i don't trust ai's answers, when i have ...",[all skills. ai is a flop.],104413.0,9.0
3,i am a developer by profession,35-44 years old,44.0,39.0,bachelor's degree,"independent contractor, freelancer, or self-em...",[none of the above],10.0,"yes, i am not new to coding but am learning ne...",[online courses or certification (includes all...,...,[],[],[],[],"[chatgpt, claude code, github copilot, google ...",[],"[when i don't trust ai's answers, when i have ...","[understand how things actually work, problem ...",53061.0,8.0
4,i am a developer by profession,35-44 years old,44.0,39.0,bachelor's degree,employed,[none of the above],4.0,"yes, i am not new to coding but am learning ne...","[ai codegen tools or ai-enabled apps, other on...",...,[],[],[],[],"[chatgpt, claude code]",[],"[when i don't trust ai's answers, when i have ...",[],36197.0,6.0
5,i am a developer by profession,35-44 years old,44.0,39.0,master's degree,"independent contractor, freelancer, or self-em...","[caring for dependents (children, elderly, etc.)]",21.0,"no, i am not new to coding and did not learn n...",[],...,[],[],[],[],[],[],[when i don't trust ai's answers],"[critical thinking, the skill to define the ta...",60000.0,7.0


In [15]:
# Zeige nur die Spalten 'YearsCode' und 'MaxAge' an
filtered_df[['WorkExp', 'MaxAge']]

,WorkExp,MaxAge
ResponseId,,
1,8.0,34.0
2,2.0,34.0
3,10.0,44.0
4,4.0,44.0
5,21.0,44.0
...,...,...
49119,9.0,34.0
49120,13.0,44.0
49121,2.0,34.0


## Filter die Zeilen raus, die mehr Coding-Erfahrung haben als Lebensalter - 6 Jahre

In [16]:
# Keep only rows where WorkExp is NOT greater than (MaxAge - 16)
filtered_df = df[~(df['YearsCode'] > (df['MaxAge'] - 6))]

# Show the number of rows before and after filtering
print(f"Original number of rows: {len(df)}")
print(f"Number of rows after filtering: {len(filtered_df)}")
print(f"Removed {len(df) - len(filtered_df)} rows ({(1 - len(filtered_df)/len(df))*100:.1f}%)")

# Display the first few rows of the filtered dataframe
filtered_df.head()

Original number of rows: 47803
Number of rows after filtering: 47723
Removed 80 rows (0.2%)


,MainBranch,Age,MaxAge,AgeNum,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
ResponseId,,,,,,,,,,,,,,,,,,,,,
1,i am a developer by profession,25-34 years old,34.0,29.0,master's degree,employed,"[caring for dependents (children, elderly, etc.)]",8.0,"yes, i am not new to coding but am learning ne...",[online courses or certification (includes all...,...,[vertex ai],[],[],[],[chatgpt],[],[when i don't trust ai's answers],"[troubleshooting, profiling, debugging]",61256.0,10.0
2,i am a developer by profession,25-34 years old,34.0,29.0,associate degree,employed,[],2.0,"yes, i am not new to coding but am learning ne...","[books / physical media, online courses or cer...",...,[],[],[],[],[],[],"[when i don't trust ai's answers, when i have ...",[all skills. ai is a flop.],104413.0,9.0
3,i am a developer by profession,35-44 years old,44.0,39.0,bachelor's degree,"independent contractor, freelancer, or self-em...",[none of the above],10.0,"yes, i am not new to coding but am learning ne...",[online courses or certification (includes all...,...,[],[],[],[],"[chatgpt, claude code, github copilot, google ...",[],"[when i don't trust ai's answers, when i have ...","[understand how things actually work, problem ...",53061.0,8.0
4,i am a developer by profession,35-44 years old,44.0,39.0,bachelor's degree,employed,[none of the above],4.0,"yes, i am not new to coding but am learning ne...","[ai codegen tools or ai-enabled apps, other on...",...,[],[],[],[],"[chatgpt, claude code]",[],"[when i don't trust ai's answers, when i have ...",[],36197.0,6.0
5,i am a developer by profession,35-44 years old,44.0,39.0,master's degree,"independent contractor, freelancer, or self-em...","[caring for dependents (children, elderly, etc.)]",21.0,"no, i am not new to coding and did not learn n...",[],...,[],[],[],[],[],[],[when i don't trust ai's answers],"[critical thinking, the skill to define the ta...",60000.0,7.0


In [17]:
filtered_df.to_csv("survey_results_shortened.csv", index=False)

In [18]:
# Zeige nur die Spalten 'YearsCode' und 'MaxAge' an
filtered_df[['YearsCode', 'MaxAge']]

,YearsCode,MaxAge
ResponseId,,
1,14.0,34.0
2,10.0,34.0
3,12.0,44.0
4,5.0,44.0
5,22.0,44.0
...,...,...
49119,13.0,34.0
49120,15.0,44.0
49121,NaN,34.0
